# Trajectory Analysis for Incremental Training

In [1]:
0

0

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import logging
import subprocess
import gc
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
import pickle
import itertools
import torch
import tqdm
import scvi
import h5py
from scipy import stats

sc.settings.verbose = 3

In [ ]:
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)
sys.path.append("/home/icb/kemal.inecik/work/codes/sctram/reproducibility/server_sync/experiments")

from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.generate.real import sc_suo_developmental_complete
from sctram.input import InputTrajectories

from helper.constants import metrics_scib, metric_direction_dict
from helper.plot_metrics_gridspec import plot_metrics_gridspec

In [ ]:
# Important to have consistent figures across platforms

%matplotlib inline
%config InlineBackend.figure_format='retina'

import pickle

from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
from adjustText import adjust_text  
import matplotlib.patheffects as path_effects

_rcparams_path = os.path.join(working_directory, "reproducibility/figure_rcparams/rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

In [ ]:
print(f"CUDA used: {torch.cuda.is_available()}")

dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

In [ ]:
definitions_path = os.path.join(dataset_dir, f"incremental_analysis_definitions_dict.pkl")
with open(definitions_path, "rb") as _file_definitions:
    definitions = pickle.load(_file_definitions)    

In [ ]:
definitions

# Loading the benchmarking

In [ ]:
sort_columns_by = ["run_id", "trajectory", "training_step", "path", "metric", "score"]

In [ ]:
df_sctram = pd.DataFrame()
count = 0

for run_id in definitions.keys():

    pickle_paths_path = definitions[run_id]["pickle_paths_path"]
    with open(pickle_paths_path, "rb") as _file_pickle_paths:
        steps_paths_dict = pickle.load(_file_pickle_paths)    
    trajectory_path = definitions[run_id]["trajectory_object_path"]
    
    for latent_step, latent_object_path in steps_paths_dict.items():
        
        with open(trajectory_path, "rb") as _file_trajectory_path:
            trajectories_object = pickle.load(_file_trajectory_path)
        
        for trajectory in sorted(trajectories_object.graph["trajectories"]):
            
            output_file_path = os.path.join(dataset_dir, f"metric_sctram_{run_id}_step_{latent_step}_trajectory_{trajectory}.pkl")
            df_trajectory_obsm = pd.read_pickle(output_file_path)
            df_trajectory_obsm["run_id"] = run_id
            df_trajectory_obsm["training_step"] = latent_step
            df_trajectory_obsm["trajectory"] = trajectory
            df_sctram = pd.concat([df_sctram, df_trajectory_obsm])
            count += 1

print(f" - Number of jobs obtained: {count}")

df_sctram.reset_index(drop=True, inplace=True)
df_sctram['score'] = pd.to_numeric(df_sctram['score'], errors='raise')
assert set(df_sctram["metric"].unique()) == set(metric_direction_dict.keys())

df_sctram = df_sctram[sort_columns_by]
df_sctram.sort_values(by=sort_columns_by, inplace=True, ignore_index=True)
df_sctram.reset_index(drop=True, inplace=True)

df_sctram

In [ ]:
df_scib = pd.DataFrame()
count = 0

for run_id in definitions.keys():

    pickle_paths_path = definitions[run_id]["pickle_paths_path"]
    with open(pickle_paths_path, "rb") as _file_pickle_paths:
        steps_paths_dict = pickle.load(_file_pickle_paths)    
    trajectory_path = definitions[run_id]["trajectory_object_path"]
    
    for latent_step, latent_object_path in steps_paths_dict.items():
        
        with open(trajectory_path, "rb") as _file_trajectory_path:
            trajectories_object = pickle.load(_file_trajectory_path)
        
        trajectories = sorted(trajectories_object.graph["trajectories"])
        trajectories += ["all_data_included"]
        
        for trajectory in trajectories:
            
            output_file_path = os.path.join(dataset_dir, f"metric_scib_{run_id}_step_{latent_step}_trajectory_{trajectory}.pkl")
            
            scib_experiment_name = f"{trajectory}_{latent_step}"
            df_scib_experiment = pd.read_pickle(output_file_path)
            df_scib_experiment = pd.DataFrame(df_scib_experiment.loc[scib_experiment_name])
            try:
                total_scib = df_scib_experiment.loc["Batch correction"][scib_experiment_name] * 0.4 + df_scib_experiment.loc["Bio conservation"][scib_experiment_name] * 0.6
            except KeyError:
                total_scib = 0.0
            df_scib_experiment['metric'] = df_scib_experiment.index
            
            df_scib_experiment.rename(columns={scib_experiment_name: "score"}, inplace=True)
            df_scib_experiment["path"] = [metrics_scib[i] for i in df_scib_experiment["metric"]]
            scib_total_dict = {"score": total_scib, "metric": "Total scIB", "path": metrics_scib["Total scIB"]}
            df_scib_experiment = pd.concat([df_scib_experiment, pd.DataFrame(pd.Series(scib_total_dict)).T], axis=0)
            df_scib_experiment.reset_index(drop=True, inplace=True)
            
            df_scib_experiment["run_id"] = run_id
            df_scib_experiment["training_step"] = latent_step
            df_scib_experiment["trajectory"] = trajectory
            df_scib = pd.concat([df_scib, df_scib_experiment])
            count += 1

print(f" - Number of jobs obtained: {count}")
df_scib.reset_index(drop=True, inplace=True)
df_scib['score'] = pd.to_numeric(df_scib['score'], errors='raise')

df_scib = df_scib[sort_columns_by]
df_scib.sort_values(by=sort_columns_by, inplace=True, ignore_index=True)
df_scib.reset_index(drop=True, inplace=True)

df_scib

# Analysis

### Bar/Box Plots etc?

In [ ]:
pass


### Progress Line Plots

In [ ]:
df_scib_by_dataset = {run_id: group.copy() for run_id, group in df_scib.groupby('run_id')}
df_sctram_by_dataset = {run_id: group.copy() for run_id, group in df_sctram.groupby('run_id')}
df_sctram_by_dataset.keys()

In [ ]:
for look_for_dataset in df_sctram_by_dataset.keys():
    
    print("\n\n\n\n")
    print("##################\n", look_for_dataset, "\n##################")
    print("\n\n\n\n")
    
    df_sctram_pivot = df_sctram_by_dataset[look_for_dataset].pivot(index=["trajectory", "path", "metric"], columns='training_step', values='score')
    df_scib_pivot = df_scib_by_dataset[look_for_dataset].pivot(index=["trajectory", "path", "metric"], columns='training_step', values='score')
    
    trajectories_sctram = list(df_sctram_by_dataset[look_for_dataset]["trajectory"].unique())
    trajectories_scib = list(df_scib_by_dataset[look_for_dataset]["trajectory"].unique())
    
    plot_metrics_gridspec(
        df_scib_pivot, 
        fig_height_per_metric=5,
        n_col=4, 
        x_axis_log_scale=True, 
        drop_zero=True, 
        trajectory=trajectories_scib,
        x_range = None,
        second_line_label_suffix="group"
    )
    
    plot_metrics_gridspec(
        df_sctram_pivot, 
        fig_height_per_metric=5,
        n_col=4, 
        x_axis_log_scale=True, 
        drop_zero=True, 
        trajectory=trajectories_sctram,
        x_range = None,
        second_line_label_suffix="group"
    )